In [ ]:
## Updated 2-18-26

## Multi-Image Spectral Standardization & K-Means Clustering

import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path(os.getcwd()).parent))


import numpy as np
import matplotlib.pyplot as plt
import tifffile
from sklearn.decomposition import PCA, IncrementalPCA
import gc

from tqdm import tqdm
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
import glob

import shutil

In [ ]:

def normalize(array, max_val=None, min_val=None, axis=None):
    """
    Normalize array to specified range.
    
    Args:
        array: Input array (1D or 2D) - numpy array
        max_val: Maximum value for normalized output (computed from data if None)
        min_val: Minimum value for normalized output (computed from data if None)
        axis: For 2D arrays only - axis along which to normalize
              axis=0: normalize each column, axis=1: normalize each row
              axis=None: global normalization
              For 1D arrays, axis parameter is ignored
    
    Returns:
        Normalized numpy array
    """
    array = np.asarray(array, dtype=np.float32)

    # For 1D arrays, ignore axis parameter
    if array.ndim == 1:
        axis = None
    
    # Calculate min/max if not provided
    if axis is None:
        # Global normalization
        if max_val is None:
            max_val = np.max(array)
            max_val = max_val.astype(np.float32)
        if min_val is None:
            min_val = np.min(array)
            min_val = min_val.astype(np.float32)
        
        diff = (max_val - min_val).astype(np.float32)
        normalized_spectra = (array - min_val) / diff + 1e-6

        return normalized_spectra
    else:
        # Axis-based normalization for 2D arrays
        if max_val is None:
            max_val = np.max(array, axis=axis, keepdims=True)
            max_val = max_val.astype(np.float32)
        if min_val is None:
            min_val = np.min(array, axis=axis, keepdims=True)
            min_val = min_val.astype(np.float32)
        
        diff = (max_val - min_val).astype(np.float32)
        return (array - min_val) / diff + 1e-6


def spectral_standardization(data, wavenum_1, wavenum_2, num_samp, background, ch_start=None):
    """
    Apply spectral standardization to hyperspectral data.
    
    Parameters
    ----------
    data : numpy.ndarray of shape (N, num_samp)
        Hyperspectral data where N is number of pixels
    wavenum_1 : float
        Starting wavenumber
    wavenum_2 : float
        Ending wavenumber
    num_samp : int
        Number of samples
    background : numpy.ndarray of shape (num_samp,)
        Background spectrum
    ch_start : int, optional
        Channel index for silent region
    
    Returns
    -------
    spectra : numpy.ndarray of shape (N, num_samp)
        Standardized spectra
    """
    if ch_start is None:
        ch_start = int((2800 - wavenum_1) / (wavenum_2 - wavenum_1) * num_samp)
    
    # Normalize input data
    data = np.asarray(data, dtype=np.float32)
    
    # Validate shape
    if data.shape[1] != num_samp:
        raise ValueError(
            f"Data channel mismatch: Expected {num_samp} channels, got {data.shape[1]}. "
            f"Check that the correct image files are being loaded. "
            f"Image shape: {data.shape}"
        )
    
    temp_norm = normalize(data)
    
    # Extract tail and head regions
    temp_end = temp_norm[:, -1:-4:-1]
    temp_start = temp_norm[:, :ch_start]
    
    # Remove baseline from silent region
    temp = temp_norm-np.mean(temp_start,axis=1)[:, np.newaxis]
    
    # Estimate background magnitude
    spectra_magnitude = np.mean(temp_end, axis=1) - np.mean(temp_start, axis=1)
    background_arr = np.outer(spectra_magnitude, background)
    
    # Subtract background
    spectra_standard = (temp - background_arr).astype(np.float32)
    
    # Normalize to background-removed spectrum
    spectra_max_idx = np.argmax(np.mean(spectra_standard, axis=0))
    max_val = np.mean(spectra_standard[:, spectra_max_idx]) + 3 * np.std(spectra_standard[:, spectra_max_idx]).astype(np.float32)  
    min_val = np.mean(spectra_standard[:, :ch_start]) - 3 * np.std(spectra_standard[:, :ch_start]).astype(np.float32)
    spectra_norm = normalize(
        spectra_standard,
        max_val=max_val,
        min_val=min_val,
        axis=1
    )
    
    # # Final baseline removal
    # spectra = spectra_norm - np.median(spectra_norm[:, :ch_start], axis=1)[:, np.newaxis]
    
    return spectra_norm.astype(np.float32)


def standardize_and_save_images(image_paths, wn_1, wn_2, num_samp, background, ch_start, output_dir=r"clustering/standardized_data"):
    """
    Standardize images and save to disk to avoid memory issues.
    Also returns metadata about pixel counts per image.
    
    Parameters
    ----------
    image_paths : list
        List of image file paths
    output_dir : str
        Directory to save standardized data
    
    Returns
    -------
    standardized_info : dict
        Dictionary with 'paths', 'pixels', and 'shapes' for each standardized image
    """
    os.makedirs(output_dir, exist_ok=True)
    
    standardized_info = {
        "name": [],
        "paths": [],
        "pixels": [],
        "shapes": []
    }
    
    total_pixels = 0
    
    for img_idx, img_path in enumerate(tqdm(image_paths, desc="Standardizing and saving images")):
        print(f"\nProcessing image {img_idx}: {os.path.basename(img_path)}")
        
        # Load and reshape image
        image_memmap = tifffile.memmap(img_path, mode='r')
        
        # Handle different TIFF formats
        # If shape is (height, width, channels), transpose to (channels, height, width)
        if image_memmap.ndim == 3:
            if image_memmap.shape[0] > image_memmap.shape[2]:  # Likely (height, width, channels)
                image_memmap = np.transpose(image_memmap, (2, 0, 1))
        
        # Reshape: (channels, height, width) -> (height*width, channels)
        reshaped = image_memmap.reshape((image_memmap.shape[0], -1)).T
        reshaped = np.asarray(reshaped)
        
        # Flip to match original orientation
        raw_data = np.flip(reshaped, axis=1)
        print(f"  Shape: {raw_data.shape}")
        
        # Standardize spectra for this image
        standardized = spectral_standardization(raw_data, wn_1, wn_2, num_samp, background, ch_start)
        
        # Save standardized data to disk
        output_path = os.path.join(output_dir, f"standardized_{img_idx:03d}.npy")
        np.save(output_path, standardized.astype(np.float32))
        
        num_pixels = standardized.shape[0]
        height, width = image_memmap.shape[1:]
        
        standardized_info["name"].append(os.path.basename(img_path))
        standardized_info["paths"].append(output_path)
        standardized_info["pixels"].append(num_pixels)
        standardized_info["shapes"].append((height, width))
        
        total_pixels += num_pixels
        
        print(f"  Standardized shape: {standardized.shape}")
    
    print(f"\nTotal pixels standardized and saved: {total_pixels:,}")
    return standardized_info
    

# def spectral_sample(info):
#     """
#     Sample spectra from standardized images and create chunks for clustering.
    
#     Parameters
#     ----------
#     info : dict
#         Dictionary with 'paths' and 'pixels' for each standardized image
#     num_samples_per_image : int
#         Number of spectra to sample from each image
    
#     Returns
#     -------
#     sampled_array : numpy.ndarray
#     """
#     sample_pixels = []
#     for file_path in info["paths"]:
#         data_mmap = np.load(file_path, mmap_mode='r')
#         sample_pixels.append(np.array(data_mmap[:]))
#         del data_mmap
#         gc.collect()
#     sample_array = np.vstack(sample_pixels)

#     return sample_array.astype(np.float32)

# def PCA_batch(sample_array, n_components=10):
#     """
#     Apply PCA to sample array for dimensionality reduction.
    
#     Parameters
#     ----------
#     sample_array : numpy.ndarray
#         Array of shape (N, num_samp) with sampled spectra
#     n_components : int
#         Number of PCA components to retain
#     Returns
#     -------
#     pca_array : numpy.ndarray
#         PCA-transformed array of shape (N, n_components)
#     """       
#     pca = PCA(n_components=n_components, random_state=42)
#     pca_data = pca.fit_transform(sample_array)
#     return pca_data.astype(np.float32)


def batch_cluster_model(init_ar=None, n_clusters=10, batch_size=10000, random_state=42):
    """
    Apply MiniBatchKMeans clustering to sample array.
    
    Parameters
    ----------
    init_ar : numpy.ndarray, optional
        Initial cluster centers for KMeans
    n_clusters : int
        Number of clusters for KMeans
    batch_size : int
        Batch size for MiniBatchKMeans
    random_state : int
        Random state for reproducibility
    
    Returns
    -------
    kmeans_model : sklearn.cluster.MiniBatchKMeans
        The fitted KMeans model
    """
    if init_ar is not None:
        kmeans_model = MiniBatchKMeans(n_clusters=n_clusters, batch_size=batch_size, random_state=random_state, init=init_ar)
    else:
        kmeans_model = MiniBatchKMeans(n_clusters=n_clusters, batch_size=batch_size, random_state=random_state, init='k-means++')

    return kmeans_model


def load_batch_data(info, batch_size=50000):
    """
    Generator to load standardized data in batches for clustering.
    Yields batches of spectra from the standardized images to reduce memory laod.
    Parameters
    ----------
    info : dict
        Dictionary with 'paths' and 'pixels' for each standardized image
    batch_size : int
        Number of spectra to yield in each batch
    Returns
    -------
    batch_data : numpy.ndarray
        Batch of spectra for clustering
    """
    buffer = []  # Buffer to accumulate pixels across batches
    for file_path in info["paths"]:
        data_mmap = np.load(file_path, mmap_mode='r')
        num_pixels = data_mmap.shape[0]
        chunk_size = min(batch_size * 2, 100000)  # Process in reasonable chunks
        for chunk_start in range(0, num_pixels, chunk_size):
            chunk_end = min(chunk_start + chunk_size, num_pixels)
            chunk = np.array(data_mmap[chunk_start:chunk_end])
            buffer.append(chunk)
            while len(buffer) > 0:
                combined = np.vstack(buffer) if len(buffer) > 1 else buffer[0]
                if combined.shape[0] >= batch_size:
                    num_full_batches = combined.shape[0] // batch_size
                    for batch_idx in range(num_full_batches):
                        start = batch_idx * batch_size
                        end = start + batch_size
                        yield combined[start:end]
                    remainder = combined[num_full_batches * batch_size:]
                    if len(remainder) > 0:
                        buffer = [remainder]
                    else:
                        buffer = []
                    break
                else:
                    break
            del chunk
            gc.collect()
        del data_mmap
        gc.collect()
    if len(buffer) > 0:
        remaining = np.vstack(buffer) if len(buffer) > 1 else buffer[0]
        if remaining.shape[0] > 0:
            yield remaining

def cluster_spectra(kmeans_model, info, batch_size=50000):
    """
    Cluster spectra using the provided KMeans model and standardized data.
    
    Parameters
    ----------
    kmeans_model : sklearn.cluster.MiniBatchKMeans
        The fitted KMeans model
    info : dict
        Dictionary with 'paths' and 'pixels' for each standardized image
    batch_size : int
        Number of spectra to process in each batch
    
    Returns
    -------
    all_labels : numpy.ndarray
        Cluster labels for all spectra across images
    """

    total_pixels = sum(info["pixels"])

    # First pass: Fit KMeans model in batches
    for batch_data in tqdm(load_batch_data(info, batch_size), desc="Fitting batches"):
        non_zero_mask = np.where(np.std(batch_data, axis=1) > 1e-2)
        batch_data = batch_data[non_zero_mask]
        kmeans_model.partial_fit(batch_data)
        del batch_data
        gc.collect()
    
    # Second pass: Predict cluster labels in batches
    labels = np.zeros(total_pixels, dtype=np.int32)
    pixel_count = 0
    for batch_data in tqdm(load_batch_data(info, batch_size), desc='CLustering batches'):
        non_zero_mask = np.where(np.std(batch_data, axis=1) > 1e-2)
        batch_non_bg = batch_data[non_zero_mask]
        batch_data = batch_non_bg[pixel_count:pixel_count + batch_size]
        labels[pixel_count:pixel_count + batch_size][non_zero_mask] = kmeans_model.predict(batch_non_bg)
        pixel_count += batch_size
        del batch_data
        gc.collect()
    
    centers = kmeans_model.cluster_centers_

    return labels, centers


def cluster_normalized_spectra(kmeans_model, info, reorg_labels, ch_start, batch_size=50000):
    """
    Re-normalize all non-background spectra (where reorg_labels != 0) in batches.
    Each spectrum is normalized individually based on its own statistics.
    
    Parameters
    ----------
    kmeans_model : MiniBatchKMeans
        KMeans model to fit on normalized data
    info : dict
        Dictionary with paths and pixels for each image
    reorg_labels : ndarray
        Current cluster labels (0 is background)
    wn_1, wn_2 : float
        Wavenumber range
    num_samp : int
        Number of spectral channels
    ch_start : int
        Channel index for silent region
    batch_size : int
        Batch size for processing
    
    Returns
    -------
    norm_labels : ndarray
        New cluster labels from re-normalized data
    norm_centers : ndarray
        New cluster centers
    """
    total_pixels = sum(info["pixels"])
    
    # First pass: Fit KMeans on normalized, non-background spectra
    print("\nFirst pass: Fitting KMeans on re-normalized spectra...")
    pixel_count = 0
    for batch_data in tqdm(load_batch_data(info, batch_size), desc="Fitting on normalized spectra"):
        batch_size_actual = batch_data.shape[0]
        batch_labels = reorg_labels[pixel_count:pixel_count + batch_size_actual]
        
        # Filter to non-background pixels
        non_bg_mask = batch_labels != 0
        if np.any(non_bg_mask):
            batch_non_bg = batch_data[non_bg_mask]
            
            # Normalize each spectrum individually using vectorized operations
            max_vals = np.max(batch_non_bg, axis=1, keepdims=True)
            min_vals = np.mean(batch_non_bg[:, :ch_start], axis=1, keepdims=True)
            
            # Normalize using per-spectrum max/min
            batch_normalized = normalize(batch_non_bg, max_val=max_vals, min_val=min_vals, axis=1)
            kmeans_model.partial_fit(batch_normalized)
            
            del batch_non_bg, batch_normalized, max_vals, min_vals
        
        pixel_count += batch_size_actual
        del batch_data
        gc.collect()
    
    # Second pass: Predict labels on normalized spectra
    print("\nSecond pass: Predicting labels on normalized spectra...")
    norm_labels = np.zeros(total_pixels, dtype=np.int32)
    pixel_count = 0
    for batch_data in tqdm(load_batch_data(info, batch_size), desc="Predicting normalized clusters"):
        batch_size_actual = batch_data.shape[0]
        batch_labels_prev = reorg_labels[pixel_count:pixel_count + batch_size_actual]
        
        # Filter to non-background pixels
        non_bg_mask = batch_labels_prev != 0
        if np.any(non_bg_mask):
            batch_non_bg = batch_data[non_bg_mask]
            
            # Normalize each spectrum individually using vectorized operations
            max_vals = np.max(batch_non_bg, axis=1, keepdims=True)
            min_vals = np.mean(batch_non_bg[:, :ch_start], axis=1, keepdims=True)
            
            batch_normalized = normalize(batch_non_bg, max_val=max_vals, min_val=min_vals, axis=1)
            predicted_labels = kmeans_model.predict(batch_normalized)
            norm_labels[pixel_count:pixel_count + batch_size_actual][non_bg_mask] = predicted_labels
            
            del batch_non_bg, batch_normalized, max_vals, min_vals
        
        pixel_count += batch_size_actual
        del batch_data
        gc.collect()
    
    norm_centers = kmeans_model.cluster_centers_
    return norm_labels, norm_centers

def reorganize_labels_clusters(labels, centers):
    """
    Reorganize cluster labels based on cluster center similarity to create a more interpretable order.
    Parameters
    ----------
    labels : numpy.ndarray
        Cluster labels for all spectra
    centers : numpy.ndarray
        Cluster centers from KMeans model
    Returns
    -------
    reorg_labels : numpy.ndarray
        Reorganized cluster labels
    reorg_centers : numpy.ndarray
        Reorganized cluster centers
    """
    # Reorganize clusters based on similarity of centers (e.g., by standard deviation or mean)
    reorg_idx = np.argsort(np.std(centers, axis=1))
    reorg_centers = centers[reorg_idx]
    reorg_labels = np.zeros_like(labels)
    for new_idx, old_idx in enumerate(reorg_idx):
        reorg_labels[labels == old_idx] = new_idx

    return reorg_labels, reorg_centers

def plot_cluster_spectra(num_clusters, reorg_centers, wn_1, wn_2, num_samp, file_name, save_dir):
    """
    Choose a color palette for visualizing clusters.
    
    Parameters
    ----------
    num_clusters : int
        Number of clusters to visualize
    reorg_centers : numpy.ndarray
        Reorganized cluster centers for plotting
    wn_1 : float
        Starting wavenumber
    wn_2 : float
        Ending wavenumber
    num_samp : int
        Number of samples (channels)
    Returns
    -------
    colors : list
        List of color codes for each cluster
    """
    
    default_colors = [
            '#000000', '#FF0000', '#00FF00', '#0000FF', 
            '#FFFF00', '#FF00FF', '#00FFFF', '#FFA500',
            '#800080', '#008080'
        ]
    
    print(f"\nSelect hexadecimal colors for cluster centers:")
    wavenumbers = np.linspace(wn_1, wn_2, num_samp)
    fig, ax = plt.subplots(figsize=(12, 6))
    color_list = []
    for i in range(num_clusters):
        default_color = default_colors[i % len(default_colors)]
        user_input = input(f'  Cluster {i} (default: {default_color}): ').strip()
        color = user_input if user_input else default_color
        color_list.append(color)
        ax.plot(wavenumbers, reorg_centers[i], color=color, linewidth=2, label=f'Cluster {i}')
    ax.set_xlabel('Wavenumber (cm$^{-1}$)')
    ax.set_xlabel('Wavenumbers (cm⁻¹)', fontsize=16)
    ax.set_ylabel('Normalized Intensity (A.U.)', fontsize=16)
    ax.set_title('K-Means Cluster Centers', fontsize=18)
    ax.legend(loc='best', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    filename = file_name
    plt.savefig(os.path.join(save_dir,filename), dpi=300)
    print(f"Saved cluster centers plot as '{filename}'")
    print(f"\nCluster colors assigned:")
    for i, color in enumerate(color_list):
        print(f"  Cluster {i}: {color}")

    # Ask user if they want to change colors
    response = input(f"\nDo you want to reassign colors? (yes/no) (View current cluster colors in {save_dir}): ").strip().lower()
    
    if response in ('yes', 'y'):
        print("\nEnter new hexadecimal colors for each cluster (or press Enter to keep current):")
        new_color_list = []
        for i in range(num_clusters):
            user_input = input(f'  Cluster {i} (current: {color_list[i]}): ').strip()
            new_color = user_input if user_input else color_list[i]
            new_color_list.append(new_color)
        
        # Display new colors
        fig, ax = plt.subplots(figsize=(12, 6))
        print("\nNew cluster center plot with updated colors:")
        for i in range(num_clusters):
            ax.plot(wavenumbers, reorg_centers[i], color=new_color_list[i], linewidth=2, label=f'Cluster {i}')
        
        ax.set_xlabel('Wavenumbers (cm⁻¹)', fontsize=12)
        ax.set_ylabel('Normalized Intensity (A.U.)', fontsize=12)
        ax.set_title('Cluster Centers - Updated Colors', fontsize=14)
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        filename = file_name
        plt.savefig(os.path.join(save_dir,filename), dpi=300)
        print(f"Saved cluster centers plot as '{filename}'")
        print(f"\nCluster colors assigned:")
        for i, color in enumerate(new_color_list):
            print(f"  Cluster {i}: {color}")

    else:
        new_color_list = color_list

    return new_color_list

def reconstruct_cluster_images(reorg_labels, info, color_list, batch_size=50000):
    """
    Reconstruct and colorize clustered images using batches.
    Uses same batch size as clustering for consistency.
    
    Parameters
    ----------
    labels : ndarray
        Cluster labels for each pixel
    image_dict : dict
        Dictionary with name, pixels, shape for each image
    color_list : list
        Hex color codes for each cluster
    batch_size : int
        Batch size (should match clustering batch size)
    
    Returns
    -------
    reconstructed_images : list
        List of RGB reconstructed images
    """
    def hex_to_rgb(hex_color):
        hex_color = hex_color.lstrip('#')
        return np.array([int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4)], dtype=np.float32)
    
    color_map = np.array([hex_to_rgb(c) for c in color_list])
    reconstructed_images = [None] * len(info['name'])
    
    print("Reconstructing images in batches...")
    total_pixels = len(reorg_labels)
    
    for batch_start in tqdm(range(0, total_pixels, batch_size), desc="Reconstructing images"):
        batch_end = min(batch_start + batch_size, total_pixels)
        batch_labels = reorg_labels[batch_start:batch_end]
        
        # Map labels to RGB colors
        batch_rgb = color_map[batch_labels.astype(int)]
        
        # Distribute batch pixels to corresponding images
        current_pixel = batch_start
        for img_idx in range(len(info['name'])):
            img_pixel_start = sum(info['pixels'][:img_idx])
            img_pixel_end = img_pixel_start + info['pixels'][img_idx]
            
            # Check overlap between batch and image
            overlap_start = max(current_pixel, img_pixel_start)
            overlap_end = min(batch_end, img_pixel_end)
            
            if overlap_start < overlap_end:
                # Initialize image if needed
                if reconstructed_images[img_idx] is None:
                    height, width = info['shapes'][img_idx]
                    reconstructed_images[img_idx] = np.zeros((height * width, 3), dtype=np.float32)
                
                # Copy RGB data to image
                batch_offset_start = overlap_start - batch_start
                batch_offset_end = overlap_end - batch_start
                img_offset_start = overlap_start - img_pixel_start
                img_offset_end = overlap_end - img_pixel_start
                
                reconstructed_images[img_idx][img_offset_start:img_offset_end] = batch_rgb[batch_offset_start:batch_offset_end]
    
    # Reshape to 2D images
    final_images = []
    for img_idx, img_pixels in enumerate(reconstructed_images):
        height, width = info['shapes'][img_idx]
        img_2d = img_pixels.reshape((height, width, 3))
        final_images.append(img_2d)
    
    return final_images

def display_reconstructed_images(reconstructed_images, info, save_dir, figsize_per_image=(5, 5), ):
    """
    Display reconstructed clustered images.
    
    Parameters
    ----------
    reconstructed_images : list of ndarray
        List of RGB images from reconstruct_images()
    info : dict
        Dictionary with image metadata
    figsize_per_image : tuple
        Figure size per image
    """
    num_images = len(reconstructed_images)
    cols = min(3, num_images)
    rows = (num_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_image[0], rows * figsize_per_image[1]))
    
    if num_images == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    for img_idx, image_2d in enumerate(reconstructed_images):
        ax = axes[img_idx]
        ax.imshow(image_2d, aspect='auto', interpolation='nearest')
        ax.set_title(f'{info["name"][img_idx]} - Clustered', fontsize=12)
        ax.set_xlabel('Width (pixels)')
        ax.set_ylabel('Height (pixels)')
    
    # Hide unused subplots
    for idx in range(num_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'clustered_images.png'), dpi=300)
    plt.show()
    
    print(f"Displayed {num_images} clustered images")



In [ ]:
# ==================== Configuration ====================
# Hyperspectral image parameters
wn_1 = 2700
wn_2 = 3100
num_samp = 61
ch_start = int((2800 - wn_1) / (wn_2 - wn_1) * num_samp)


# Data directories
base_dir = os.path.dirname(os.getcwd())
clustering_dir = os.path.join(base_dir, 'clustering')
image_dir = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium HSI\ref"
srs_params_path = os.path.join(base_dir, 'params_dataset', 'srs_params_61.npz')

# Standardized data temp folder in clustering
standardized_dir = os.path.join(clustering_dir, 'standardized_data')

# Output images/plots folder: clustering/<basename(image_dir)>_output_images
output_base = os.path.basename(os.path.normpath(image_dir)) + '_output_images'
output_dir = os.path.join(clustering_dir, output_base)
os.makedirs(output_dir, exist_ok=True)

# Load background from SRS parameters
srs_data = np.load(srs_params_path)
background_spectrum = srs_data['background']

In [ ]:
# ==================== Main Workflow ====================
# Print configuration
print("\n" + "="*60)
print("Configuration:")
print(f"  Wavenumber range: {wn_1}-{wn_2} cm⁻¹")
print(f"  Number of samples: {num_samp}")
print(f"  Channel start: {ch_start}")
print(f"  Background spectrum shape: {background_spectrum.shape}")
print("="*60 + "\n")

# Find images
image_paths = sorted(glob.glob(os.path.join(image_dir, '*.tif')))

if not image_paths:
    print(f"ERROR: No .tif files found in {image_dir}")
    raise FileNotFoundError(f"No .tif files found in {image_dir}")

print(f"Found {len(image_paths)} images")



In [ ]:
# ==================== STANDARDIZATION (Save to Disk) ====================
print("Standardizing images and saving to disk...")


standardized_info = standardize_and_save_images(
    image_paths, wn_1, wn_2, num_samp, background_spectrum, ch_start,
    output_dir=standardized_dir
)

# Visualize standardized spectra from batch for sanity check
# Use for loop instead of generator assignment to properly clean up resources
sample_batch = None
for batch_data in load_batch_data(standardized_info, batch_size=10000):
    sample_batch = batch_data
    break  # Only need first batch for visualization

del batch_data
gc.collect()

condition= np.std(sample_batch, axis=1) > 1e-2

sample_batch = sample_batch[np.where(condition)]  # Filter out spectra with no variation

print(f"\nSample batch shape for visualization: {sample_batch.shape}")

random_indices = np.random.choice(sample_batch.shape[0], size=min(5, sample_batch.shape[0]), replace=False)
wavenumbers = np.linspace(wn_1, wn_2, num_samp)
plt.figure(figsize=(12, 6))
for idx in random_indices:
    plt.plot(wavenumbers, sample_batch[idx], alpha=0.7, label=f'Sample {idx}')
plt.title('Sample Standardized Spectra from Batch', fontsize=18)
plt.xlabel('Wavenumbers (cm⁻¹)', fontsize=16)
plt.ylabel('Normalized Intensity (A.U.)', fontsize=16)
plt.legend()    

In [ ]:
# ==================== Un-normalized Clustering ====================
n_clusters = int(input(f"Enter number of clusters for k-means (or press Enter to use 7): ") or 7)
kmeans_model = batch_cluster_model(n_clusters=n_clusters, batch_size=10000, random_state=42)

batch_size = 50000
labels, centers = cluster_spectra(
    kmeans_model=kmeans_model,
    info=standardized_info,
    batch_size=batch_size
)
reorg_labels, reorg_centers = reorganize_labels_clusters(labels, centers)

# Plot cluster spectra
color_list = plot_cluster_spectra(
    num_clusters=n_clusters,
    reorg_centers=reorg_centers,
    wn_1=wn_1,
    wn_2=wn_2,
    num_samp=num_samp,
    file_name='kmeans_cluster_centers.png',
    save_dir=output_dir
)

# Reconstruct clusters into 2D images
images = reconstruct_cluster_images(
    reorg_labels=reorg_labels,
    info=standardized_info,
    color_list=color_list,
    batch_size=batch_size
)

display_reconstructed_images(images, standardized_info, save_dir=output_dir, figsize_per_image=(6, 6))
print(f"Successfully reconstructed {len(images)} initial clustered images")

# Save RGB clustered images as individual tif files in output_dir
for idx, img in enumerate(images):
    img_name = os.path.splitext(os.path.basename(standardized_info['name'][idx]))[0]
    out_path = os.path.join(output_dir, f"{img_name}_clustered.tif")
    tifffile.imwrite(out_path, (img * 255).astype(np.uint8))
    print(f"Saved clustered image: {out_path}")

# Save kmeans centroid plot to output_dir (move if needed)
centroid_plot_path = os.path.join(output_dir, 'kmeans_cluster_centers.png')
if os.path.exists(centroid_plot_path):
    shutil.move(centroid_plot_path, os.path.join(output_dir, 'kmeans_cluster_centers.png'))
    print(f"Saved centroid plot: {os.path.join(output_dir, 'kmeans_cluster_centers.png')}")


print("\nWorkflow completed successfully!")
print(f"Clustered images and centroid plot saved to '{output_dir}'")

In [ ]:

# ==================== Normalized CLustering ====================
# Create new KMeans model and re-cluster normalized  spectra

print("Normalizing non-background spectra and re-clustering...")


# Create new KMeans model with same parameters
kmeans_model_norm = batch_cluster_model(n_clusters=n_clusters, batch_size=10000, random_state=42)

# Re-normalize and re-cluster
norm_labels_unnorm, norm_centers_unnorm = cluster_normalized_spectra(
    kmeans_model=kmeans_model_norm,
    info=standardized_info,
    reorg_labels=reorg_labels,
    wn_1=wn_1,
    wn_2=wn_2,
    num_samp=num_samp,
    ch_start=ch_start,
    batch_size=50000
)

# Reorganize labels
norm_labels, norm_centers = reorganize_labels_clusters(norm_labels_unnorm, norm_centers_unnorm)


# Plot re-normalized cluster centers and select colors
color_list_norm = plot_cluster_spectra(
    num_clusters=n_clusters,
    reorg_centers=norm_centers,
    wn_1=wn_1,
    wn_2=wn_2,
    num_samp=num_samp,
    file_name='kmeans_cluster_centers_normalized.png',
    save_dir=output_dir
)

# Reconstruct normalized cluster images
images_norm = reconstruct_cluster_images(
    reorg_labels=norm_labels,
    info=standardized_info,
    color_list=color_list_norm,
    batch_size=50000
)

display_reconstructed_images(images_norm, standardized_info, save_dir=output_dir, figsize_per_image=(6, 6))
print(f"Successfully reconstructed {len(images_norm)} re-normalized clustered images")

# Save re-normalized clustered images as individual tif files
print("\nSaving re-normalized clustered images...")
for idx, img in enumerate(images_norm):
    img_name = os.path.splitext(os.path.basename(standardized_info['name'][idx]))[0]
    out_path = os.path.join(output_dir, f"{img_name}_normalized_clustered.tif")
    tifffile.imwrite(out_path, (img * 255).astype(np.uint8))
    print(f"Saved normalized clustered image: {out_path}")


In [ ]:
# Clean up: delete standardized_data temp folder
if os.path.exists(standardized_dir):
    shutil.rmtree(standardized_dir)
    print(f"Deleted temporary folder: {standardized_dir}")
